# পাঠ ১৭ - Foundry Local এবং Qwen দিয়ে স্থানীয় AI এজেন্ট তৈরি করা

এই নোটবুকে আপনি একটি **স্থানীয় ইঞ্জিনিয়ারিং সহকারী** তৈরি করবেন যা সম্পূর্ণরূপে আপনার ওয়ার্কস্টেশনে চলে। কোনও সময়ে ক্লাউড ইনফারেন্স ব্যবহৃত হয় না। সহকারীটি করতে পারে:

১। **Qwen ফাংশন কলিং এর মাধ্যমে Foundry Local এ টুলস কল করা।**
২। **স্যান্ডবক্সড প্রোজেক্ট ডিরেক্টরির ভিতরে ফাইল তালিকা ও পড়া।**
৩। **সহজ মেট্রিক্স দিয়ে কোড বিশ্লেষণ করা।**
৪। **লোকাল RAG (Chroma) দিয়ে ডকুমেন্টেশন সার্চ করা।**
৫। **একটি স্থানীয় MCP সার্ভার ব্যবহার করা** (কনফিগার করা না থাকলে সাবধানে এড়ানো হয়)।

এজেন্ট কোডটি প্রায় একই রকম ক্লাউড পাঠের মতো — শুধুমাত্র ক্লায়েন্ট এন্ডপয়েন্ট ক্লাউড থেকে `localhost` এ চলে যায়।


## Setup

Before running this notebook:

1. **Install Microsoft Foundry Local** (see the [documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) for your OS).
2. **Download and start a Qwen model:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Install the Python packages below.

Everything runs locally. A machine with ~8 GB RAM is a realistic minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## ১. ফাউন্ড্রি লোকাল এর সাথে সংযোগ করুন

`FoundryLocalManager` প্রয়োজনে মডেল ডাউনলোড করে, লোকাল সার্ভিস শুরু করে, এবং আমাদের একটি **OpenAI-সামঞ্জস্যপূর্ণ এন্ডপয়েন্ট** প্রদান করে। তারপর আমরা স্ট্যান্ডার্ড OpenAI SDK-কে সেটির দিকে নির্দেশ করি। API কী একটি লোকাল প্লেসহোল্ডার — কোনও ক্লাউড ক্রেডেনশিয়াল জড়িত নয়।


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. লোকাল টুলস (স্যান্ডবক্সড ফাইল অপারেশনস)

আমরা ডিস্কে একটি ছোট নমুনা প্রকল্প তৈরি করি, তারপর সেই প্রকল্প রুটের জন্য সীমিত টুলস নির্ধারণ করি। স্যান্ডবক্স চেক লোকাল হলেও গুরুত্বপূর্ণ: একটি টুল যা যেকোনো পথ পড়ে, আপনার ব্যবহারকারীর অনুমতিতে চলে এবং আপনি যা স্পর্শ করতে পারেন, সেটি স্পর্শ করতে পারে।


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. লোকাল RAG উইথ ক্রোমা

আমরা স্থানীয় ক্রোমা সংগ্রহে একটি ছোট ডকুমেন্টেশন স্নিপেট সেট এমবেড করি। ক্রোমা ইন-প্রোসেসে চলে এবং ডিস্কে ভেক্টর সংরক্ষণ করে — কোনও সার্ভার নেই, কোনও ক্লাউড নেই। `search_docs` টুলটি একটি প্রশ্নের জন্য সবচেয়ে প্রাসঙ্গিক স্নিপেটগুলি পুনরুদ্ধার করে।


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## ৪. টুল-কলিং লুপ

এখন আমরা OpenAI টুলস স্কিমা ব্যবহার করে মডেলের সাথে টুলগুলি নিবন্ধন করি এবং স্ট্যান্ডার্ড টুল-কলিং লুপ চালাই — মডেল একটি টুল অনুরোধ করে, আমরা সেটি স্থানীয়ভাবে সম্পাদন করি, ফলাফলটি ফিরে দিই, এবং পুনরাবৃত্তি করি যতক্ষণ না মডেল একটি চূড়ান্ত উত্তর তৈরি করে। ক্বেনের নির্ভরযোগ্য ফাংশন কলিংই এই কাজটি ডিভাইসে করতে সক্ষম করে।  


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## ৫. লোকাল MCP (ঐচ্ছিক)

MCP একটি ট্রান্সপোর্ট, ক্লাউড সার্ভিস নয় — একটি MCP সার্ভার লোকাল প্রসেস হিসেবে `stdio`-এর মাধ্যমে চালানো যেতে পারে। নিচের সেলে দেখানো হয়েছে কিভাবে আপনি একটি লোকাল MCP সার্ভারের সাথে সংযুক্ত হবেন যদি এটি কনফিগার করা থাকে (যেমন একটি ফাইলসিস্টেম সার্ভার)। এটি সুন্দরভাবে এড়িয়ে যায় যখন `LOCAL_MCP_COMMAND` সেট করা না থাকে, তাই নোটবুকটি তবুও শুরু থেকে শেষ পর্যন্ত চলতে থাকে।

নিরাপত্তা নোট: একটি লোকাল MCP সার্ভার আপনার ব্যবহারকারীর অনুমতিতে চলে। এটিকে একটি প্রজেক্ট ডিরেক্টরির মধ্যে সীমাবদ্ধ করুন এবং এর আউটপুট যাচাই করে তারপরে কাজ করুন।


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## সারাংশ

আপনি একটি ইঞ্জিনিয়ারিং সহকারী তৈরি করেছেন যা সম্পূর্ণরূপে আপনার মেশিনে চলে:

- **Foundry Local** একটি **Qwen** মডেল একটি OpenAI-সঙ্গত সংযোগের পেছনে সরবরাহ করেছিল — তাই এজেন্ট কোড ক্লাউড পাঠের সাথে মেলে।
- **Sandboxed tools** এজেন্টকে একটি প্রকল্প ডিরেক্টরি ছাড়াই ফাইল অ্যাক্সেস এবং কোড বিশ্লেষণ দিয়েছে।
- **Chroma** ডকুমেন্টেশনের উপর **স্থানীয় RAG** প্রদান করেছিল।
- **Local MCP** অফলাইনে MCP ইকোসিস্টেম পুনরায় ব্যবহার করার পদ্ধতি দেখিয়েছে।

কোনও পয়েন্টে ক্লাউড ইনফারেন্স ব্যবহৃত হয়নি।

### চ্যালেঞ্জ

স্থানীয় এজেন্টকে সম্প্রসারিত করুন:

1. **অনেক MCP সার্ভারের সাথে কাজ করুন** — একটি ফাইলসিস্টেম সার্ভার এবং একটি গিট সার্ভারের সাথে সংযোগ স্থাপন করুন এবং এজেন্টকে তাদের মধ্যে বেছে নিতে দিন।
2. **স্থানীয় মেমোরি ব্যবহার করুন** — ডিস্কে সংক্ষিপ্ত কথোপকথন ইতিহাস সংরক্ষণ করুন যাতে সহকারী নোটবুক পুনরায় চালু হলে আগের রাউন্ডগুলি মনে রাখতে পারে।
3. **স্থানীয় বহু-এজেন্ট সমন্বয় সমর্থন করুন** — দ্বিতীয় একটি স্থানীয় এজেন্ট (যেমন একটি পর্যালোচক) যোগ করুন এবং দুইজনকে একটি কাজ নিয়ে সহযোগিতা করতে দিন।

পরবর্তী পাঠে আপনি শিখবেন কিভাবে মোতায়েনকৃত AI এজেন্টদের নিরাপদ করবেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
